In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

## Loading Training and testing data

#### Training Data

In [2]:
training_data={
    "Glioma":tf.data.Dataset.list_files("Tumor_dataset/Training/glioma/*").take(1200),
    "Meningioma":tf.data.Dataset.list_files("Tumor_dataset/Training/meningioma/*").take(1200),
    "Pituitary":tf.data.Dataset.list_files("Tumor_dataset/Training/pituitary/*").take(1200),
    "No_Tumor":tf.data.Dataset.list_files("Tumor_dataset/Training/notumor/*").take(1200)
}

#### Testing Data

In [3]:
testing_data={
    "Glioma":tf.data.Dataset.list_files("Tumor_dataset/Testing/glioma/*").take(300),
    "Meningioma":tf.data.Dataset.list_files("Tumor_dataset/Testing/meningioma/*").take(300),
    "Pituitary":tf.data.Dataset.list_files("Tumor_dataset/Testing/pituitary/*").take(300),
    "No_Tumor":tf.data.Dataset.list_files("Tumor_dataset/Testing/notumor/*").take(300)
}

# Prepare Data Pipeline

### Helper Functions

In [4]:
import os

In [5]:
def get_label(file_path):
    class_name = tf.strings.split(file_path, os.path.sep)[-2]
    if class_name == tf.constant("notumor"):
        return 0
    elif class_name == tf.constant("glioma"):
        return 1
    elif class_name == tf.constant("meningioma"):
        return 2
    else:
        return 3



In [16]:
def process_image(file_path):
    label=get_label(file_path)
    img=tf.io.read_file(file_path)
    img=tf.image.decode_jpeg(img)
    if img.shape[-1]==1:
        img=tf.tile(img,[1,1,3])
    img=tf.image.convert_image_dtype(img,tf.float32)
    img=img/255.0
    img=tf.image.resize(img,[240,240])

    return img,label

### Mapping fuctions to pipeline and preparing pipeline by concatenating all classes

In [17]:
train_ds=training_data['Glioma'].concatenate(training_data['Meningioma'].concatenate(training_data['No_Tumor'].concatenate(training_data['Pituitary']))).map(process_image)
test_ds=testing_data['Glioma'].concatenate(testing_data['Meningioma'].concatenate(testing_data['No_Tumor'].concatenate(testing_data['Pituitary']))).map(process_image)

#### Checking count of gray scale and color images

In [18]:
gray=0
color=0
for img,label in train_ds:
    if(img.shape[-1]==1):
        gray+=1
    else:
        color+=1
        
print(gray)
print(color)

2190
2610


## Creating batches

In [16]:
batch_size=200

In [17]:
train_ds=train_ds.batch(batch_size)
test_ds=test_ds.batch(batch_size)

In [18]:
# Iterate through the train_ds and print out the shapes of images and labels
for images, labels in train_ds.take(1):  # Take one batch for inspection
    print("Image shape:", images.shape)
    print("Label shape:", labels.shape)


InvalidArgumentError: {{function_node __wrapped__IteratorGetNext_output_types_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} Cannot add tensor to the batch: number of elements does not match. Shapes are: [tensor]: [240,240,3], [batch]: [240,240,1] [Op:IteratorGetNext] name: 

In [13]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming your test_ds elements have images and labels
# Adjust the code based on your actual dataset structure

# Initialize a dictionary to store class counts
class_counts = {'class_0': 0, 'class_1': 0, 'class_2': 0, 'class_3': 0}

# Iterate through the test_ds and count occurrences of each class
for images, labels in train_ds:
    for label in labels.numpy():
        class_name = f'class_{label}'
        class_counts[class_name] += 1

# Print the class counts
for class_name, count in class_counts.items():
    print(f'{class_name}: {count}')

# Plot class distribution
class_labels, counts = zip(*class_counts.items())
plt.bar(class_labels, counts)
plt.xlabel('Class')
plt.ylabel('Number of Samples')
plt.title('Class Distribution in Train Dataset')
plt.show()


TypeError: 'numpy.int32' object is not iterable

In [40]:
model =Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(240, 240, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(240, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(training_data), activation='softmax')
])

In [41]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_6 (Conv2D)           (None, 238, 238, 32)      896       
                                                                 
 max_pooling2d_6 (MaxPoolin  (None, 119, 119, 32)      0         
 g2D)                                                            
                                                                 
 conv2d_7 (Conv2D)           (None, 117, 117, 64)      18496     
                                                                 
 max_pooling2d_7 (MaxPoolin  (None, 58, 58, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_8 (Conv2D)           (None, 56, 56, 128)       73856     
                                                                 
 max_pooling2d_8 (MaxPoolin  (None, 28, 28, 128)      

In [42]:
early_stopping=EarlyStopping(monitor='val_loss',#preventing overfitting
                             patience=3,
                             restore_best_weights=True)


In [43]:

history=model.fit(train_ds,
                  epochs=10,
                  validation_data=(test_ds),
                  callbacks=[early_stopping])

Epoch 1/10


InvalidArgumentError: Graph execution error:

Detected at node 'IteratorGetNext' defined at (most recent call last):
    File "<frozen runpy>", line 198, in _run_module_as_main
    File "<frozen runpy>", line 88, in _run_code
    File "C:\Users\Acer\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
      app.launch_new_instance()
    File "C:\Users\Acer\AppData\Local\Programs\Python\Python311\Lib\site-packages\traitlets\config\application.py", line 1043, in launch_instance
      app.start()
    File "C:\Users\Acer\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\kernelapp.py", line 728, in start
      self.io_loop.start()
    File "C:\Users\Acer\AppData\Local\Programs\Python\Python311\Lib\site-packages\tornado\platform\asyncio.py", line 195, in start
      self.asyncio_loop.run_forever()
    File "C:\Users\Acer\AppData\Local\Programs\Python\Python311\Lib\asyncio\base_events.py", line 607, in run_forever
      self.